# UHI Classification — Refined Baseline (v2)

Refinements in this version:

1. **Bugs fixed** — osmnx imported at the top; feature-importance cell
   corrected; city references replaced by soft-coded bounding boxes.
2. **Soft-coded geocoding** — road features now build from a lat/lon bbox
   derived from the point data, not from place-name queries.  This removes
   the Freetown geocoding failure.
3. **Length-weighted road features** — the traffic proxy now uses road
   length × hierarchy weight rather than a raw count.
4. **Multi-scale expansion** — `BUFFER_SCALES_M = [30, 100, 250]`.
5. **Per-prediction confidence-weighted ensemble** — each model's
   contribution is weighted by its own max-probability on that specific
   point, so highly confident models dominate contentious predictions.
6. **Pseudo-labeled Freetown rows added to training** — points the previous
   model predicted correctly are merged back into training data with their
   true UHI_Class from the validation CSV.  **This changes what "validation"
   means — see the data-leakage note in that section.**
7. **Per-model predictions in the output CSV** — so you can see where models
   agreed or disagreed.
8. **Feature correlation diagnostics** — heatmap of a curated subset plus
   a table of the top correlated feature pairs.

In [ ]:
# ---------------------------------------------------------------------------
# Clone the bc-II repo for UHI CSVs and install requirements
# ---------------------------------------------------------------------------
!git clone https://github.com/chase-kusterer/bc-II.git
%cd bc-II
!cat .env_bin.txt > /dev/null
!pip install -r /content/bc-II/requirements.txt

In [ ]:
# Mount Drive
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# Install all extras in one go — imports below will fail fast if missing
!pip install rioxarray xgboost lightgbm catboost osmnx networkx

In [ ]:
# Suppress non-critical warnings
import warnings
warnings.filterwarnings('ignore')

# File management and tabular data
import os
import numpy as np
import pandas as pd

# Geospatial raster / vector I/O
import rioxarray as rxr
import rasterio
import geopandas as gpd
from shapely.geometry import Point, box as shapely_box

# OSM road network (imported at top — fixes earlier NameError)
import osmnx as ox
import networkx as nx

# Visualisation
import matplotlib.pyplot as plt
import seaborn as sns

# Modelling
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_validate
from sklearn.preprocessing import LabelEncoder
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score, classification_report,
    confusion_matrix, ConfusionMatrixDisplay,
)

# Progress bar
from tqdm import tqdm

## Configuration

All paths, feature toggles, and ensemble settings live here.  The key new
path is `FREETOWN_CORRECT_CSV` — the file of previously correctly-predicted
Freetown points to be merged into the training set.

In [ ]:
# ===========================================================================
# TRAINING REGIONS
# ===========================================================================
INCLUDE_CHILE  = True
INCLUDE_BRAZIL = True

# Pseudo-labeled Freetown points (from previous model's correct predictions)
INCLUDE_FREETOWN_CORRECT = True
FREETOWN_CORRECT_CSV = "/content/bc-II/Data/team1_correct_predictions.csv"  # Path to CSV; adjust as needed

# ===========================================================================
# INPUT FILE PATHS
# ===========================================================================
DRIVE_ROOT = "/content/drive/MyDrive/Business Challenge II"
TIF_DIR    = f"{DRIVE_ROOT}/GeoTIFFs"
BD_DIR     = f"{DRIVE_ROOT}/BuildingDensity"

# --- Chile / Santiago ------------------------------------------------------
CHILE_UHI_CSV       = "/content/bc-II/Data/sample_chile_uhi_data.csv"
CHILE_GEOTIFF_PATH  = f"{TIF_DIR}/Santiago_MBR(2023-12-20TO2024-02-20)_Median_AllBands.tiff"
CHILE_DENSITY_CACHE = f"{BD_DIR}/Chile_building_density_100m.parquet"

# --- Brazil / Rio de Janeiro ----------------------------------------------
BRAZIL_UHI_CSV       = "/content/bc-II/Data/sample_brazil_uhi_data.csv"
BRAZIL_GEOTIFF_PATH  = f"{TIF_DIR}/Rio_MBR(2024-06-11)_Median_AllBands.tiff"
BRAZIL_DENSITY_CACHE = f"{BD_DIR}/Rio_building_density_100m.parquet"

# --- Validation / Freetown -------------------------------------------------
VAL_UHI_CSV       = "/content/bc-II/Data/validation_dataset.csv"
VAL_GEOTIFF_PATH  = f"{TIF_DIR}/Freetown_MBR(2023-01-31)_Median_AllBands.tiff"
VAL_DENSITY_CACHE = f"{BD_DIR}/SierraLeone_building_density_100m.parquet"

# ===========================================================================
# FEATURE TOGGLES
# ===========================================================================
USE_EXTENDED_INDICES    = True
BUFFER_SCALES_M         = [30, 100, 250]    # multi-scale expansion (suggestion #4)
USE_MORPHOLOGY_FEATURES = False
USE_ROAD_FEATURES       = True              # newly soft-coded (suggestion #1 + bbox)
ROAD_BUFFER_M           = 500
BBOX_PADDING_DEG        = 0.02              # padding around data bbox for OSM download

# ===========================================================================
# ENSEMBLE CONFIGURATION
# ===========================================================================
ENSEMBLE_MODELS = ["rf", "extratrees", "xgboost", "lightgbm", "logreg", "knn", "mlp"]

# ===========================================================================
# OUTPUTS
# ===========================================================================
SUBMISSION_CSV = "/content/Predicted_Dataset.csv"

# ---------------------------------------------------------------------------
# Sanity-check required inputs
# ---------------------------------------------------------------------------
required = []
if INCLUDE_CHILE:
    required += [CHILE_UHI_CSV, CHILE_GEOTIFF_PATH, CHILE_DENSITY_CACHE]
if INCLUDE_BRAZIL:
    required += [BRAZIL_UHI_CSV, BRAZIL_GEOTIFF_PATH, BRAZIL_DENSITY_CACHE]
required += [VAL_UHI_CSV, VAL_GEOTIFF_PATH, VAL_DENSITY_CACHE]
if INCLUDE_FREETOWN_CORRECT:
    required.append(FREETOWN_CORRECT_CSV)

missing = [p for p in required if not os.path.exists(p)]
if missing:
    print("Missing required inputs:")
    for p in missing:
        print(f"  {p}")
    raise FileNotFoundError("Fix config paths before proceeding.")
print("All required inputs present.")
print(f"Scales: {BUFFER_SCALES_M} | Roads: {USE_ROAD_FEATURES} | "
      f"Freetown-correct injection: {INCLUDE_FREETOWN_CORRECT}")

## Feature Extraction Helpers

The spectral / buffer / building helpers are unchanged from the previous
notebook.  The road-traffic helper has been rewritten:

* **Soft-coded geocoding**: uses a lat/lon bbox derived from the points
  themselves (plus padding).  No place-name strings.
* **Length-weighted traffic score**: sum of `(segment length × hierarchy
  weight)` instead of a count of road segments.

In [ ]:
# ===========================================================================
# Helper — open a GeoTIFF with band-name metadata attached
# ===========================================================================
def open_geotiff(path):
    data = rxr.open_rasterio(path, masked=True)
    with rasterio.open(path) as src:
        descs = src.descriptions
        if descs and any(d for d in descs):
            data = data.assign_coords(band_name=("band", list(descs)))
    return data

In [ ]:
# ===========================================================================
# Helper — per-point buffered pixel extraction
# ===========================================================================
def extract_buffered_features(geotiff_path, df, buffer_m,
                              lat_col="Latitude", lon_col="Longitude"):
    data = open_geotiff(geotiff_path)
    band_names = (
        list(data.coords["band_name"].values)
        if "band_name" in data.coords
        else [f"band{int(b)}" for b in data["band"].values]
    )
    # Half-window in degrees (assumes CRS is EPSG:4326)
    half_deg = (buffer_m / 2.0) / 111320.0

    records = []
    for lat, lon in tqdm(
        zip(df[lat_col].values, df[lon_col].values),
        total=len(df),
        desc=f"Buffer {buffer_m}m features",
    ):
        window = data.sel(
            x=slice(lon - half_deg, lon + half_deg),
            y=slice(lat + half_deg, lat - half_deg),   # y descending
        )
        if window.sizes["x"] == 0 or window.sizes["y"] == 0:
            window = data.sel(x=lon, y=lat, method="nearest").expand_dims("x").expand_dims("y")

        rec = {}
        for i, bname in enumerate(band_names):
            arr = window.isel(band=i).values.astype("float64")
            valid = arr[~np.isnan(arr)] if arr.size else arr
            if valid.size == 0:
                m = s = mn = mx = np.nan
            else:
                m, s = valid.mean(), valid.std()
                mn, mx = valid.min(), valid.max()
            rec[f"{bname}_{buffer_m}m_mean"] = m
            rec[f"{bname}_{buffer_m}m_std"]  = s
            rec[f"{bname}_{buffer_m}m_min"]  = mn
            rec[f"{bname}_{buffer_m}m_max"]  = mx
        records.append(rec)
    return pd.DataFrame(records)

In [ ]:
# ===========================================================================
# Helper — per-scale spectral indices from buffered band means
# ===========================================================================
def add_spectral_indices(feat_df, buffer_m, extended=True):
    def c(name):
        col = f"{name}_{buffer_m}m_mean"
        return feat_df[col] if col in feat_df.columns else None
    b02, b03, b04, b08, b11 = c("B02"), c("B03"), c("B04"), c("B08"), c("B11")

    if all(x is not None for x in (b03, b04, b08, b11)):
        feat_df[f"NDVI_{buffer_m}m"] = (b08 - b04) / (b08 + b04 + 1e-9)
        feat_df[f"NDBI_{buffer_m}m"] = (b11 - b08) / (b11 + b08 + 1e-9)
        feat_df[f"NDWI_{buffer_m}m"] = (b03 - b08) / (b03 + b08 + 1e-9)

    if extended and all(x is not None for x in (b02, b03, b04, b08, b11)):
        feat_df[f"EVI_{buffer_m}m"]   = 2.5 * (b08 - b04) / (b08 + 6*b04 - 7.5*b02 + 1 + 1e-9)
        feat_df[f"SAVI_{buffer_m}m"]  = 1.5 * (b08 - b04) / (b08 + b04 + 0.5 + 1e-9)
        feat_df[f"MNDWI_{buffer_m}m"] = (b03 - b11) / (b03 + b11 + 1e-9)
        feat_df[f"BSI_{buffer_m}m"]   = (
            ((b11 + b04) - (b08 + b02)) / ((b11 + b04) + (b08 + b02) + 1e-9)
        )
    return feat_df

In [ ]:
# ===========================================================================
# Helper — load cached building density
# ===========================================================================
def load_building_features(cache_path, use_morphology=False):
    gdf = gpd.read_parquet(cache_path)
    keep = ['building_density_100m']
    if use_morphology:
        morph_cols = [
            'mean_height_100m', 'max_height_100m', 'footprint_ratio_100m',
            'building_count_100m', 'built_volume_proxy_100m',
        ]
        keep += [c for c in morph_cols if c in gdf.columns]
    return pd.DataFrame(gdf[keep])

In [ ]:
# ===========================================================================
# Helper — road / traffic features via soft-coded bbox (no place-name geocoding)
# ===========================================================================
# Road hierarchy -> weight (higher = more traffic-like)
ROAD_WEIGHTS = {
    "motorway":  5,
    "trunk":     4,
    "primary":   3,
    "secondary": 2,
    "tertiary":  1,
}

def _road_weight(tag):
    """Turn a highway tag (may be list or string) into a numeric weight."""
    if isinstance(tag, list):
        # Take the highest-priority tag in the list
        return max((ROAD_WEIGHTS.get(t, 0) for t in tag), default=0)
    return ROAD_WEIGHTS.get(tag, 0)


def load_road_traffic_features(
    df, lat_col="Latitude", lon_col="Longitude",
    buffer_m=500, padding_deg=0.02,
):
    """
    Compute per-point road/traffic features using an OSM road network
    downloaded from a bounding box covering the points (plus padding).

    Output columns:
      dist_to_major_road_m       — nearest major-road distance (metres)
      road_density_{buffer}m     — total road length per km² within buffer
      intersection_density_{buffer}m — intersections per km² within buffer
      traffic_proxy_score_{buffer}m  — length-weighted hierarchy score within buffer

    No place-name query.  The bbox is derived from the data itself.
    """
    min_lat = df[lat_col].min() - padding_deg
    max_lat = df[lat_col].max() + padding_deg
    min_lon = df[lon_col].min() - padding_deg
    max_lon = df[lon_col].max() + padding_deg
    print(f"  OSM bbox (lat {min_lat:.4f}..{max_lat:.4f}, "
          f"lon {min_lon:.4f}..{max_lon:.4f})")

    # Download road network from OSM using the coordinate bbox.
    # osmnx API signature: graph_from_bbox(north, south, east, west, ...)
    try:
        G = ox.graph_from_bbox(
            north=max_lat, south=min_lat, east=max_lon, west=min_lon,
            network_type="drive",
        )
    except Exception as e:
        print(f"  [warning] OSM graph download failed: {e}")
        print(f"  Returning NaN road features — model will see these as missing.")
        n = len(df)
        return pd.DataFrame({
            "dist_to_major_road_m":                  [np.nan] * n,
            f"road_density_{buffer_m}m":             [np.nan] * n,
            f"intersection_density_{buffer_m}m":     [np.nan] * n,
            f"traffic_proxy_score_{buffer_m}m":      [np.nan] * n,
        })

    nodes, edges = ox.graph_to_gdfs(G)
    edges["road_weight"] = edges["highway"].apply(_road_weight)
    major_edges = edges[edges["road_weight"] >= ROAD_WEIGHTS["tertiary"]].copy()

    # Build a points GeoDataFrame and convert to metric CRS for distance/area
    pts = gpd.GeoDataFrame(
        df.copy(),
        geometry=gpd.points_from_xy(df[lon_col], df[lat_col]),
        crs="EPSG:4326",
    ).to_crs(epsg=3857)
    edges = edges.to_crs(epsg=3857)
    major_edges = major_edges.to_crs(epsg=3857)
    nodes = nodes.to_crs(epsg=3857)

    # Pre-compute edge lengths for length-weighted traffic score
    edges["seg_length_m"] = edges.geometry.length

    area_km2 = (np.pi * (buffer_m ** 2)) / 1_000_000

    records = []
    for _, row in tqdm(pts.iterrows(), total=len(pts), desc="Road features"):
        point = row.geometry
        buf = point.buffer(buffer_m)

        # Nearest-major-road distance (skip if no major roads in region)
        if len(major_edges) > 0:
            dist_to_major = major_edges.distance(point).min()
        else:
            dist_to_major = np.nan

        # Edges intersecting the buffer
        hit = edges[edges.intersects(buf)]
        total_len = hit["seg_length_m"].sum() if len(hit) else 0.0
        road_density = total_len / area_km2 if area_km2 else 0.0

        # Intersection density — count of OSM graph nodes inside the buffer
        intersections = nodes[nodes.intersects(buf)]
        intersection_density = len(intersections) / area_km2 if area_km2 else 0.0

        # Length-weighted traffic proxy — each intersecting segment contributes
        # (length × hierarchy weight).  This upgrades the original count-based
        # score so a long motorway counts much more than five short tertiaries.
        if len(hit):
            traffic_score = float((hit["seg_length_m"] * hit["road_weight"]).sum())
        else:
            traffic_score = 0.0

        records.append({
            "dist_to_major_road_m":                  dist_to_major,
            f"road_density_{buffer_m}m":             road_density,
            f"intersection_density_{buffer_m}m":     intersection_density,
            f"traffic_proxy_score_{buffer_m}m":      traffic_score,
        })

    return pd.DataFrame(records)

## Per-Region Feature Assembly

In [ ]:
# ===========================================================================
# Per-region pipeline — same logic for every region
# ===========================================================================
def build_region_features(
    uhi_csv, geotiff_path, density_cache, region_id,
    has_labels=True, df_override=None,
):
    """
    Build the full feature DataFrame for one region.

    If `df_override` is provided, it is used as the starting DataFrame
    instead of reading `uhi_csv` — used to process a filtered subset
    (e.g., the Freetown-correct points) without rereading the CSV.
    """
    print(f"\n=== Building features for {region_id} ===")
    df = df_override.copy() if df_override is not None else pd.read_csv(uhi_csv)
    df = df.reset_index(drop=True)
    print(f"  Points: {len(df)}")

    # --- Multi-scale buffer features ---------------------------------------
    all_scale_feats = []
    for scale in BUFFER_SCALES_M:
        feats = extract_buffered_features(geotiff_path, df, buffer_m=scale)
        feats = add_spectral_indices(feats, scale, extended=USE_EXTENDED_INDICES)
        all_scale_feats.append(feats)
    buffer_df = pd.concat(all_scale_feats, axis=1)

    # --- Building density (and optional morphology) ------------------------
    bldg_df = load_building_features(density_cache, use_morphology=USE_MORPHOLOGY_FEATURES)
    # density cache is keyed by CSV row order; if df_override is a subset of
    # the original CSV, we need to align by index
    if df_override is not None:
        bldg_df = bldg_df.iloc[df.index.values if hasattr(df.index, 'values') else range(len(df))]
        bldg_df = bldg_df.reset_index(drop=True)

    # --- Road features (soft-coded bbox) -----------------------------------
    if USE_ROAD_FEATURES:
        road_df = load_road_traffic_features(
            df, buffer_m=ROAD_BUFFER_M, padding_deg=BBOX_PADDING_DEG,
        )
    else:
        road_df = pd.DataFrame(index=range(len(df)))

    out = pd.concat(
        [df.reset_index(drop=True),
         buffer_df.reset_index(drop=True),
         bldg_df.reset_index(drop=True),
         road_df.reset_index(drop=True)],
        axis=1,
    )
    out["region"] = region_id
    return out

## Pseudo-Label Injection from Freetown "Correct" Points

The CSV `team1_correct_predictions.csv` marks which Freetown points the
previous model got right.  We merge those coordinates with the validation
CSV (which has true `UHI_Class` labels) and inject the matched rows into
the training data.

### ⚠️ Data-leakage caveat

These Freetown points come from the validation dataset.  Adding them to
training means:

* The current held-out test split **cannot** be the same validation file
  you previously compared against — doing so would test on data now in
  training.  If you want a clean out-of-sample comparison to past runs,
  predict on the *rest* of the Freetown points (the ones that were
  incorrect previously, i.e. not in this CSV).
* Interpret any accuracy gain relative to the old baseline cautiously —
  part of the gain may reflect the model having seen some of what it is
  now being tested on.

In [ ]:
# ===========================================================================
# Merge Freetown-correct coords with labels from the validation CSV
# ===========================================================================
def build_freetown_correct_training_frame():
    """
    Load the correct-prediction CSV, keep rows where Correct=True, and
    inner-merge with the validation UHI CSV (on Longitude/Latitude) to pick
    up the UHI_Class labels.  Returns a DataFrame ready to feed into
    build_region_features as a df_override.
    """
    corr = pd.read_csv(FREETOWN_CORRECT_CSV)
    print(f"  Correct-predictions CSV: {len(corr)} rows "
          f"({corr['Correct'].sum()} correct / {(~corr['Correct']).sum()} incorrect)")

    corr_true = corr[corr['Correct'] == True].reset_index(drop=True)

    # Pull labels from the Freetown validation CSV
    val = pd.read_csv(VAL_UHI_CSV)
    if 'UHI_Class' not in val.columns:
        raise ValueError(
            "Validation CSV has no UHI_Class column — cannot merge labels "
            "onto correct-prediction coordinates."
        )

    # Round coords to a precision that matches both files
    round_decimals = 6
    corr_true['_lat_r'] = corr_true['Latitude'].round(round_decimals)
    corr_true['_lon_r'] = corr_true['Longitude'].round(round_decimals)
    val['_lat_r'] = val['Latitude'].round(round_decimals)
    val['_lon_r'] = val['Longitude'].round(round_decimals)

    merged = corr_true.merge(
        val[['_lat_r', '_lon_r', 'UHI_Class']],
        on=['_lat_r', '_lon_r'],
        how='inner',
    ).drop(columns=['_lat_r', '_lon_r'])
    print(f"  Matched {len(merged)} correct Freetown points to labels "
          f"(of {len(corr_true)} 'Correct=True' rows)")

    if len(merged) < len(corr_true):
        print(f"  [note] {len(corr_true) - len(merged)} rows didn't match — "
              f"likely due to coordinate-precision differences.")

    # Keep only columns needed downstream
    return merged[['Longitude', 'Latitude', 'UHI_Class']].drop_duplicates().reset_index(drop=True)

In [ ]:
# ---------------------------------------------------------------------------
# Assemble each training region; optionally append Freetown-correct rows.
# ---------------------------------------------------------------------------
train_regions = []

if INCLUDE_CHILE:
    chile_df = build_region_features(
        CHILE_UHI_CSV, CHILE_GEOTIFF_PATH, CHILE_DENSITY_CACHE,
        region_id="chile", has_labels=True,
    )
    train_regions.append(chile_df)

if INCLUDE_BRAZIL:
    brazil_df = build_region_features(
        BRAZIL_UHI_CSV, BRAZIL_GEOTIFF_PATH, BRAZIL_DENSITY_CACHE,
        region_id="brazil", has_labels=True,
    )
    train_regions.append(brazil_df)

if INCLUDE_FREETOWN_CORRECT:
    print("\n=== Injecting Freetown correct predictions into training set ===")
    ft_correct_df = build_freetown_correct_training_frame()
    # Use df_override so we only compute features for these specific coords,
    # not the full validation CSV.
    ft_correct_features = build_region_features(
        VAL_UHI_CSV, VAL_GEOTIFF_PATH, VAL_DENSITY_CACHE,
        region_id="freetown_correct", has_labels=True,
        df_override=ft_correct_df,
    )
    train_regions.append(ft_correct_features)

if not train_regions:
    raise ValueError("At least one training region must be enabled.")

train_full = pd.concat(train_regions, axis=0, ignore_index=True)
print(f"\nCombined training shape: {train_full.shape}")
print("Class balance by region:")
print(train_full.groupby("region")["UHI_Class"].value_counts())

In [ ]:
# ---------------------------------------------------------------------------
# Identify feature columns (everything that isn't metadata)
# ---------------------------------------------------------------------------
META_COLS = {"Longitude", "Latitude", "UHI_Class", "region", "geometry", "Correct"}
feature_cols = [c for c in train_full.columns if c not in META_COLS]
print(f"Total feature count: {len(feature_cols)}")

# Drop rows with NaN in any predictor column
before = len(train_full)
train_full = train_full.dropna(subset=feature_cols).reset_index(drop=True)
print(f"Dropped {before - len(train_full)} NaN rows. Remaining: {len(train_full)}")

In [ ]:
# ===========================================================================
# Within-region z-score standardisation
# ===========================================================================
def standardise_within_region(df, feature_cols, region_col="region"):
    out = df.copy()
    for col in feature_cols:
        grp = out.groupby(region_col)[col]
        out[col] = (out[col] - grp.transform("mean")) / (grp.transform("std") + 1e-9)
    return out

train_std = standardise_within_region(train_full, feature_cols)

## Feature Correlation Analysis

Two complementary views of feature correlations:

1. A heatmap of a **curated subset** (one representative feature per family)
   so the plot is readable at a glance.
2. A **table of the top 20 correlated pairs** across the full feature set —
   pairs correlated above |0.95| are redundant and could be candidates
   for dropping if you want a leaner model.

In [ ]:
# ---------------------------------------------------------------------------
# Curated subset — pick one feature per "family" for a readable heatmap
# ---------------------------------------------------------------------------
def pick_representative(cols, pattern):
    """Return the first column containing `pattern`, or None."""
    for c in cols:
        if pattern in c:
            return c
    return None

subset_patterns = [
    "NDVI_100m", "NDBI_100m", "NDWI_100m",
    "EVI_100m", "SAVI_100m", "MNDWI_100m", "BSI_100m",
    "NDVI_30m", "NDVI_250m",
    "B08_100m_std", "B04_100m_mean", "B11_100m_mean",
    "building_density_100m",
    "dist_to_major_road_m",
    f"road_density_{ROAD_BUFFER_M}m",
    f"intersection_density_{ROAD_BUFFER_M}m",
    f"traffic_proxy_score_{ROAD_BUFFER_M}m",
]
subset = [c for c in
          (pick_representative(feature_cols, p) for p in subset_patterns)
          if c is not None]

corr_subset = train_std[subset].corr()

plt.figure(figsize=(11, 9))
sns.heatmap(corr_subset, annot=True, fmt=".2f", cmap="coolwarm",
            center=0, vmin=-1, vmax=1, square=True, cbar_kws={"shrink": 0.7})
plt.title("Feature Correlation — Representative Subset")
plt.tight_layout()
plt.show()

In [ ]:
# ---------------------------------------------------------------------------
# Full-set correlation — top 20 most correlated pairs
# Useful for spotting redundant features across the full pipeline.
# ---------------------------------------------------------------------------
full_corr = train_std[feature_cols].corr().abs()
# Zero out the diagonal and lower triangle so each pair appears once
mask = np.triu(np.ones_like(full_corr, dtype=bool), k=1)
pairs = (
    full_corr.where(mask)
    .stack()
    .sort_values(ascending=False)
    .head(20)
    .rename("abs_corr")
    .reset_index()
    .rename(columns={"level_0": "feature_a", "level_1": "feature_b"})
)
print("Top 20 correlated feature pairs:")
print(pairs.to_string(index=False))

## Train / Test Split

Stratified on UHI_Class.

In [ ]:
X = train_std[feature_cols].values
y = train_std["UHI_Class"].values

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, stratify=y, random_state=123,
)
print(f"X_train: {X_train.shape} | X_test: {X_test.shape}")

In [ ]:
# ===========================================================================
# Model factory — all members expose predict_proba for the confidence-weighted
# ensemble below
# ===========================================================================
def make_model(choice):
    if choice == "rf":
        return RandomForestClassifier(
            n_estimators=500, max_depth=20, min_samples_leaf=2,
            random_state=42, class_weight='balanced', n_jobs=-1,
        )
    if choice == "extratrees":
        from sklearn.ensemble import ExtraTreesClassifier
        return ExtraTreesClassifier(
            n_estimators=500, max_depth=20, min_samples_leaf=2,
            random_state=42, class_weight='balanced', n_jobs=-1,
        )
    if choice == "xgboost":
        from xgboost import XGBClassifier
        return XGBClassifier(
            n_estimators=500, max_depth=6, learning_rate=0.05,
            subsample=0.9, colsample_bytree=0.9,
            objective="multi:softprob", eval_metric="mlogloss",
            verbosity=0, random_state=42, n_jobs=-1,
        )
    if choice == "lightgbm":
        from lightgbm import LGBMClassifier
        return LGBMClassifier(
            n_estimators=500, max_depth=-1, learning_rate=0.05,
            num_leaves=63, subsample=0.9, colsample_bytree=0.9,
            class_weight='balanced', random_state=42, n_jobs=-1, verbose=-1,
        )
    if choice == "catboost":
        from catboost import CatBoostClassifier
        return CatBoostClassifier(
            iterations=500, depth=6, learning_rate=0.05,
            loss_function='MultiClass', auto_class_weights='Balanced',
            random_seed=42, verbose=0,
        )
    if choice == "logreg":
        from sklearn.linear_model import LogisticRegression
        return LogisticRegression(
            C=1.0, max_iter=2000, multi_class='multinomial',
            class_weight='balanced', random_state=42, n_jobs=-1,
        )
    if choice == "knn":
        from sklearn.neighbors import KNeighborsClassifier
        return KNeighborsClassifier(n_neighbors=15, weights='distance', n_jobs=-1)
    if choice == "mlp":
        from sklearn.neural_network import MLPClassifier
        return MLPClassifier(
            hidden_layer_sizes=(64, 32), activation='relu',
            alpha=1e-3, learning_rate_init=1e-3,
            max_iter=300, early_stopping=True, random_state=42,
        )
    raise ValueError(f"Unknown model choice: {choice}")

In [ ]:
# ===========================================================================
# Label-encode once, fit all ensemble members
# ===========================================================================
le = LabelEncoder()
y_train_enc = le.fit_transform(y_train)
y_test_enc  = le.transform(y_test)
class_labels = le.classes_        # e.g. ['High', 'Low', 'Medium']
n_classes = len(class_labels)

trained_models = {}
for name in ENSEMBLE_MODELS:
    print(f"Training {name}...")
    m = make_model(name)
    m.fit(X_train, y_train_enc)
    trained_models[name] = m
print(f"\nTrained {len(trained_models)} models.")

## Confidence-Weighted Soft-Vote Ensemble

Earlier versions of this ensemble used a simple average of probability
vectors — every model got an equal vote regardless of how confident it was
on a given point.  This version implements **per-prediction confidence
weighting** as you described:

> For each point, check the confidence of each model.  Models more confident
> about that specific prediction receive higher weights.

Formally, for each point:

* Each model $m$ outputs a probability vector $\mathbf{p}_m$ over classes.
* Its confidence on that point is $c_m = \max_k p_{m,k}$.
* The ensemble probability is $\frac{\sum_m c_m \mathbf{p}_m}{\sum_m c_m}$.

A model that predicts `[0.95, 0.03, 0.02]` (confidence 0.95) will dominate
the ensemble for that point, while a model that hedges with
`[0.40, 0.35, 0.25]` (confidence 0.40) contributes less than half as much.
When models agree in confidence, this reduces to the simple mean — so it's
never *worse* than equal-vote averaging.

In [ ]:
# ===========================================================================
# Per-prediction confidence weighting
# ===========================================================================
def ensemble_predict_proba_weighted(models, X):
    """
    Returns:
      ensemble_proba : (n_samples, n_classes)  — confidence-weighted average
      per_model_proba: dict[name -> (n_samples, n_classes)] for diagnostics
    """
    per_model_proba = {name: m.predict_proba(X) for name, m in models.items()}
    # Stack to (n_models, n_samples, n_classes)
    stacked = np.stack(list(per_model_proba.values()), axis=0)

    # Confidence of each model on each sample — max prob across classes.
    # Shape: (n_models, n_samples)
    confidences = stacked.max(axis=2)

    # Weight probabilities by confidence (broadcast over class axis).
    # Normalise by the sum of confidences per sample so weights are proper.
    weights_norm = confidences / (confidences.sum(axis=0, keepdims=True) + 1e-9)
    ensemble_proba = (stacked * weights_norm[:, :, None]).sum(axis=0)

    return ensemble_proba, per_model_proba


def ensemble_predict_weighted(models, X, class_labels):
    proba, _ = ensemble_predict_proba_weighted(models, X)
    return class_labels[np.argmax(proba, axis=1)]

In [ ]:
# ===========================================================================
# Compare individual models vs. weighted ensemble on the held-out test split
# ===========================================================================
print("Per-model test accuracy:")
for name, m in trained_models.items():
    preds = le.inverse_transform(m.predict(X_test))
    print(f"  {name:12s}: {accuracy_score(y_test, preds):.4f}")

ens_preds = ensemble_predict_weighted(trained_models, X_test, class_labels)
print(f"  {'ens (weighted)':12s}: {accuracy_score(y_test, ens_preds):.4f}")

print("\nWeighted ensemble classification report:")
print(classification_report(y_test, ens_preds))

In [ ]:
# ---------------------------------------------------------------------------
# Confusion matrix for the weighted ensemble on the test split
# ---------------------------------------------------------------------------
labels_sorted = sorted(np.unique(y_test))
cm = confusion_matrix(y_test, ens_preds, labels=labels_sorted)
ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=labels_sorted).plot(cmap='Blues')
plt.title('Confusion Matrix — Confidence-Weighted Ensemble')
plt.grid(False)
plt.show()

## Apply to Validation Region

Same pipeline — build features for the full Freetown validation file, run
the ensemble, and save a CSV that contains both the final prediction and
**each individual model's prediction** for diagnostics.

In [ ]:
# ---------------------------------------------------------------------------
# Build Freetown features (full validation CSV — all points)
# ---------------------------------------------------------------------------
val_df = build_region_features(
    VAL_UHI_CSV, VAL_GEOTIFF_PATH, VAL_DENSITY_CACHE,
    region_id="freetown", has_labels=False,
)

missing_cols = [c for c in feature_cols if c not in val_df.columns]
if missing_cols:
    raise ValueError(f"Validation frame is missing columns: {missing_cols}")

val_df = val_df.dropna(subset=feature_cols).reset_index(drop=True)
val_std = standardise_within_region(val_df, feature_cols)
X_val = val_std[feature_cols].values
print(f"X_val: {X_val.shape}")

In [ ]:
# ---------------------------------------------------------------------------
# Confidence-weighted ensemble prediction + per-model predictions
# ---------------------------------------------------------------------------
ens_proba, per_model_proba = ensemble_predict_proba_weighted(trained_models, X_val)
val_ensemble_pred = class_labels[np.argmax(ens_proba, axis=1)]

# Per-model predictions + per-model confidences for diagnostics
per_model_pred_cols = {}
per_model_conf_cols = {}
for name, proba in per_model_proba.items():
    per_model_pred_cols[f"pred_{name}"] = class_labels[np.argmax(proba, axis=1)]
    per_model_conf_cols[f"conf_{name}"] = proba.max(axis=1).round(4)

# Assemble submission frame with every diagnostic the user asked for
submission_df = pd.DataFrame({
    'Longitude': val_std['Longitude'].values,
    'Latitude' : val_std['Latitude'].values,
    'UHI_Class': val_ensemble_pred,
    'ensemble_confidence': ens_proba.max(axis=1).round(4),
    **per_model_pred_cols,
    **per_model_conf_cols,
})

print("Final predicted class distribution:")
print(submission_df['UHI_Class'].value_counts())
submission_df.head()

In [ ]:
# ---------------------------------------------------------------------------
# Agreement summary — how often the ensemble disagrees with individual models
# ---------------------------------------------------------------------------
print("Per-model agreement with final ensemble prediction (higher = more similar):")
for name in trained_models.keys():
    agree = (submission_df[f"pred_{name}"] == submission_df["UHI_Class"]).mean()
    print(f"  {name:12s}: {agree:.4f}")

unanimous = np.all(
    [submission_df[f"pred_{n}"] == submission_df["UHI_Class"] for n in trained_models.keys()],
    axis=0,
)
print(f"\nAll models unanimous: {unanimous.mean():.4f}")

In [ ]:
# Save the submission CSV with per-model diagnostics
submission_df.to_csv(SUBMISSION_CSV, index=False)
print(f"Submission saved to: {SUBMISSION_CSV}")
print(f"Columns: {submission_df.columns.tolist()}")

## Notes

* **If you re-submit to the same benchmark**, remember the Freetown-correct
  points are now in the training set.  The submission contains predictions
  for *all* Freetown rows, including those used for pseudo-label training.
  Consider computing a "clean" accuracy that excludes those rows before
  comparing to the previous baseline.
* **The `ensemble_confidence` column** tells you how sure the weighted
  ensemble is about each point.  Low-confidence rows (e.g., below 0.45) are
  likely in the Medium/High or Low/Medium boundary regions — worth
  inspecting spatially.
* **Inter-model disagreement columns (`pred_*`)** make it easy to identify
  points where models split.  High disagreement often correlates with
  ambiguous labels; low disagreement with high confidence is the model's
  strongest signal.